In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. Load the Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

# 2. Define Target, Leaks, and ID Column
TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'

# These are the features causing the 0.9+ R2 data leakage. 
# We MUST drop them from both the training and testing sets.
LEAKED_FEATURES = ['trainer_focus_score', 'damage_dealt', 'healing_applied']

# Prepare Training Data
y_train = train_df[TARGET]
X_train = train_df.drop(columns=[TARGET, ID_COL] + LEAKED_FEATURES)

# Prepare Testing Data (Save IDs for the final submission format)
test_ids = test_df[ID_COL]
X_test = test_df.drop(columns=[ID_COL] + LEAKED_FEATURES)

# 3. Build a Preprocessing Pipeline
# Identify which columns are text/categorical and which are numbers
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# Fill missing numbers with the median
numeric_transformer = SimpleImputer(strategy='median')

# Fill missing text with the most frequent value, then convert to numbers.
# handle_unknown='use_encoded_value' ensures the model doesn't crash if 
# the test set contains a category it never saw during training.
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 4. Define and Train the Model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

print("Training model (this might take a moment)...")
model.fit(X_train, y_train)

# 5. Predict on Test Data and Format Submission
print("Predicting on test data...")
predictions = model.predict(X_test)

# Create the final dataframe matching the sample_submission format
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: predictions
})

# Save to CSV
submission.to_csv('submissionDay9.csv', index=False)
print("Saved predictions to 'submission.csv'")

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_31412\3626102385.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training model (this might take a moment)...
Predicting on test data...
Saved predictions to 'submission.csv'
